# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane 2, Refresh / Content Opportunity Scoring.** The question is the same as in W01: which pages should a reviewer look at first? In W04 I built a simple rule (`sliding_with_demand`) and froze it as the number to beat. This notebook trains a first model and compares it with that rule on the same pages, the same five client folds and the same metric (precision@50).

Same frame as W03 and W04: **one row = one page**, features from March 2026 only and the label `declined_next_month` (April impressions per day more than 20% below March). The rule and the label are not changed.

**How to run it:** same as W03. In Colab save your Hugging Face READ token as a Secret named `HF_TOKEN`. Outside Colab the notebook reads the `HF_TOKEN` environment variable or asks for it with `getpass`. The token is never printed and never written into the notebook. Every random step uses seed 42, so a rerun gives the same tables.

In [1]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
import getpass
import platform
from datetime import date
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, ndcg_score, roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# The repo is public, so the token comes from a Colab Secret or an environment variable.
# It is never printed. getpass is only the last fallback.
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass
HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN") or getpass.getpass("Hugging Face READ token: ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
print("DuckDB connected. Hugging Face token loaded (not printed).")
VERSIONS = {"python": platform.python_version(), "scikit-learn": sklearn.__version__, "pandas": pd.__version__,
            "numpy": np.__version__, "duckdb": duckdb.__version__}
print("Versions:", " | ".join(f"{k} {v}" for k, v in VERSIONS.items()))

DuckDB connected. Hugging Face token loaded (not printed).
Versions: python 3.13.16 | scikit-learn 1.9.1 | pandas 3.0.5 | numpy 2.5.3 | duckdb 2.0.0


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**My lane is a "which first?" ranking with a label I can observe** (W02). So the right tool is a classifier: it gives every page a probability of declining next month, I sort the pages by it and judge the top of the list with precision@50, the same metric as W02 and W04. Clustering does not fit because I already have a label. The signal analysis was done in the W04 audit, so this week uses its verdicts instead of repeating them.

From this week's menu I use two models, simple first:

| Model | Why |
|---|---|
| **Logistic regression** | Readable. It gives one weight per feature, so I can check every sign against the W04 signal verdicts. With 5 features and about 65,000 to 69,000 training pages per fold it has little room to overfit |
| **Gradient boosting** (`HistGradientBoostingClassifier`) | The stronger check. It can learn bends and interactions the linear model can't, for example CTR meaning something different at each position. It is safe here because every input is known on 31 March and every score is measured on clients the model never saw. I keep its default settings (seed 42). Tuning it on the same folds I report would make it look better than it is |

**Inputs.** Both models get the **five W03 contract features** (`imp_per_day_log`, `ctr_pct`, `avg_position`, `late_march_share`, `gsc_days`) without any change. One more gradient boosting run tests the ideas the W04 audit left for this week, four extra features:

| Extra feature | How it is built | Known by 31 March because… |
|---|---|---|
| `ctr_gap` | CTR minus the median CTR of the page's position tier. The tier medians come from the training clients only | it only uses March clicks, impressions and positions |
| `odd_position` | 1 when the average position is below 1 (the odd pages W03 found) | it only uses March positions |
| `feb_per_day_log` | log(1 + February impressions per day). Missing when the page does not have all 28 February days or its client was not tracked from 1 February. Gradient boosting handles missing values itself | February ends before March starts |
| `late_vs_feb_log` | log(1 + late-March impressions per day) minus `feb_per_day_log`. Above 0 means late March is still above the February level, so a "slide" there is probably a surge ending | both parts end on or before 31 March |

**How I pick "my model":** the simplest one, unless a more complex one wins clearly on the held-out folds. One more floor sits next to the rule: **sorting by the slide alone** (`late_march_share`, no model). The slide is the strongest signal from W04, so a model is only worth it if it also beats that simple sort.

In [3]:
REL = "hf://datasets/FlyRank/internship-warehouse"

# Same contract as W03 and W04: features from March, label from April
FEATURE_MONTH = "2026-03"          # feature window: 1 to 31 March 2026
LABEL_MONTH = "2026-04"            # label window: 1 to 30 April 2026
PRE_MONTH = "2026-02"              # February: before the decision moment, only for the extra features
MARCH_DAYS, APRIL_DAYS, FEB_DAYS = 31, 30, 28
LATE_MARCH_START = "2026-03-16"    # second half of March starts here
EARLY_DAYS, LATE_DAYS = 15, 16     # 1 to 15 March and 16 to 31 March
GSC_TRACKED_BY = "2026-03-01"
MIN_MARCH_IMPRESSIONS = 100
DECLINE_CUTOFF_PCT = -20
LABEL = "declined_next_month"

# The W04 rule, unchanged
MIN_SLIDE_PCT = 20
MIN_LATE_IMPRESSIONS_PER_DAY = 5
RULE = "W04 rule (baseline)"

# This week
SEED = 42
N_FOLDS = 5
K_MAIN = 50                                       # main metric since W02: precision@50
POSITION_TIERS = [0, 1, 3, 10, 20, 50, np.inf]    # same tiers as the W04 signal audit

FEATURES_W03 = ["imp_per_day_log", "ctr_pct", "avg_position", "late_march_share", "gsc_days"]
FEATURES_EXTRA = ["ctr_gap", "odd_position", "feb_per_day_log", "late_vs_feb_log"]


def logistic_regression():
    # Readable: one weight per feature. Features are scaled first so the weights can be compared
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))


def gradient_boosting():
    # Stronger check. Default settings with a fixed seed. Nothing is tuned on the test folds
    return HistGradientBoostingClassifier(random_state=SEED)


MAIN = "logistic regression"
MODELS = {
    "logistic regression": (FEATURES_W03, logistic_regression),
    "gradient boosting": (FEATURES_W03, gradient_boosting),
    "gradient boosting + 4 extra": (FEATURES_W03 + FEATURES_EXTRA, gradient_boosting),
}
for name, (cols, _) in MODELS.items():
    print(f"{name:28} {len(cols)} features: {', '.join(cols)}")

logistic regression          5 features: imp_per_day_log, ctr_pct, avg_position, late_march_share, gsc_days
gradient boosting            5 features: imp_per_day_log, ctr_pct, avg_position, late_march_share, gsc_days
gradient boosting + 4 extra  9 features: imp_per_day_log, ctr_pct, avg_position, late_march_share, gsc_days, ctr_gap, odd_position, feb_per_day_log, late_vs_feb_log


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Same split as W04: `GroupKFold(n_splits=5)` on `client_hash_id`.** Every page in a test fold comes from clients the model never saw in training.

Why this is the honest split for my question:
- W03 showed the label moves with the client (April decline rates from 27% to 94% across the big clients). With a random row split, pages of the same client sit on both sides, so the model could learn "this client had a bad April" and look better than it is. Grouping by client asks the real question: does it work on a client it has never seen?
- It is exactly the same five folds as W04. The cells below check this against the W04 receipt: same pages, same base rate and the same rule P@50 on every fold.
- Everything learned from data is learned on the training folds only: the scaler, the model weights, the tree splits and the CTR tier medians behind `ctr_gap`.
- No time-aware split this week. My frame has one March → April pair, so there is only one time step to test. W06 asks for a re-run under a time-aware design, where I can use the February → March slice from the signal audit.

**One more design choice: the review pool.** The models are trained and scored only on pages that still had at least 5 impressions a day in late March (16 to 31 March). This is not a new rule: it is the same visibility bar the W04 rule already uses (`MIN_LATE_IMPRESSIONS_PER_DAY = 5`). A page under it has mostly vanished already and needs a technical check (removed, redirected or dropped from the index), not a refresh. The bar only uses March numbers, never the label.

### Rebuild the W03 / W04 page frame

Same query and the same four eligibility rules as W03 and W04. I only add February per page (for the extra features) and the biggest single day in early March (for the error review). April is read only for the label.

In [4]:
pages = con.sql(f'''
    WITH march_pages AS (
        SELECT
            content_hash_id,
            ANY_VALUE(client_hash_id)                           AS client_hash_id,  -- safe: W03 query 1 showed one client per page
            COUNT(*)                                            AS march_rows,
            SUM(gsc_impressions)::BIGINT                        AS imp_march,
            SUM(gsc_clicks)::BIGINT                             AS clicks_march,
            SUM(gsc_impressions) FILTER (WHERE report_date >= DATE '{LATE_MARCH_START}')::BIGINT AS imp_late_march,
            MAX(gsc_impressions) FILTER (WHERE report_date <  DATE '{LATE_MARCH_START}')::BIGINT AS early_peak_day,
            COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)  AS gsc_days,
            SUM(gsc_sum_position) FILTER (WHERE gsc_data_available IS TRUE AND gsc_avg_position > 0)::BIGINT AS position_sum,
            SUM(gsc_impressions)  FILTER (WHERE gsc_data_available IS TRUE AND gsc_avg_position > 0)::BIGINT AS position_impressions
        FROM read_parquet('{REL}/fact_content_daily_performance/month={FEATURE_MONTH}/*.parquet')
        GROUP BY content_hash_id
    ),
    april_pages AS (  -- label window only: no feature is built from this
        SELECT content_hash_id, COUNT(*) AS april_rows, SUM(gsc_impressions)::BIGINT AS imp_april
        FROM read_parquet('{REL}/fact_content_daily_performance/month={LABEL_MONTH}/*.parquet')
        GROUP BY content_hash_id
    ),
    feb_pages AS (    -- February: before the decision moment, only for the extra features
        SELECT content_hash_id, COUNT(*) AS feb_rows, SUM(gsc_impressions)::BIGINT AS imp_feb
        FROM read_parquet('{REL}/fact_content_daily_performance/month={PRE_MONTH}/*.parquet')
        GROUP BY content_hash_id
    )
    SELECT m.*, c.gsc_data_start, a.april_rows, a.imp_april, f.feb_rows, f.imp_feb
    FROM march_pages m
    LEFT JOIN read_parquet('{REL}/dim_clients.parquet') c USING (client_hash_id)
    LEFT JOIN april_pages a USING (content_hash_id)
    LEFT JOIN feb_pages f USING (content_hash_id)
''').df()
counts = [c for c in pages.columns if c not in ("content_hash_id", "client_hash_id", "gsc_data_start")]
pages[counts] = pages[counts].astype("float64")          # SQL NULL becomes NaN
pages["gsc_data_start"] = pd.to_datetime(pages["gsc_data_start"])

# The four W03 eligibility rules
keep = (pages["march_rows"].eq(MARCH_DAYS)
        & pages["gsc_data_start"].le(pd.Timestamp(GSC_TRACKED_BY))
        & pages["imp_march"].ge(MIN_MARCH_IMPRESSIONS)
        & pages["april_rows"].eq(APRIL_DAYS))
e = pages[keep].sort_values("content_hash_id").reset_index(drop=True)

frame = pd.DataFrame({
    "content_hash_id": e["content_hash_id"],
    "client_hash_id": e["client_hash_id"],
    # the five W03 features (March only)
    "imp_per_day_log": np.log1p(e["imp_march"] / MARCH_DAYS),
    "ctr_pct": 100 * e["clicks_march"] / e["imp_march"],
    "avg_position": e["position_sum"] / e["position_impressions"],
    "late_march_share": e["imp_late_march"] / e["imp_march"],
    "gsc_days": e["gsc_days"],
    # W04 rule inputs and review columns (March only)
    "imp_per_day": e["imp_march"] / MARCH_DAYS,
    "early_per_day": (e["imp_march"] - e["imp_late_march"]) / EARLY_DAYS,
    "late_per_day": e["imp_late_march"] / LATE_DAYS,
    "clicks_per_day": e["clicks_march"] / MARCH_DAYS,
})
frame["slide_pct"] = 100 * (1 - frame["late_per_day"] / frame["early_per_day"].where(frame["early_per_day"] > 0))
early_imp = e["imp_march"] - e["imp_late_march"]
frame["early_peak_share"] = e["early_peak_day"] / early_imp.where(early_imp > 0)

# Extra features. ctr_gap is added inside each fold, because its tier medians must come from training clients only
frame["odd_position"] = (frame["avg_position"] < 1).astype("float64")
full_feb = e["feb_rows"].eq(FEB_DAYS) & e["gsc_data_start"].le(pd.Timestamp(f"{PRE_MONTH}-01")) & e["imp_feb"].gt(0)
frame["feb_per_day"] = (e["imp_feb"] / FEB_DAYS).where(full_feb)       # NaN when February is not complete
frame["feb_per_day_log"] = np.log1p(frame["feb_per_day"])
frame["late_vs_feb_log"] = np.log1p(frame["late_per_day"]) - frame["feb_per_day_log"]   # above 0 = late March still above February
frame["feb_rows"] = e["feb_rows"].fillna(0)        # review only: on how many February days the page had a row

# The label: April vs March per day (same as W03). The % change stays OUT of the frame, it is the answer
april_per_day = e["imp_april"] / APRIL_DAYS
april_vs_march_pct = 100 * (april_per_day - frame["imp_per_day"]) / frame["imp_per_day"]
frame[LABEL] = (april_vs_march_pct < DECLINE_CUTOFF_PCT).astype(int)
hindsight = pd.DataFrame({"april_per_day": april_per_day.to_numpy(), "april_vs_march_pct": april_vs_march_pct.to_numpy()},
                         index=e["content_hash_id"])   # only read after scoring, for the error review

# Client rank: 1 = client with the most pages (same order as W03 / W04). Hash ids stay out of every display
client_pages = frame.groupby("client_hash_id").size().reset_index(name="n").sort_values(["n", "client_hash_id"], ascending=[False, True])
client_rank = pd.Series(np.arange(1, len(client_pages) + 1), index=client_pages["client_hash_id"].to_numpy())
frame["client_rank"] = frame["client_hash_id"].map(client_rank)

assert frame["content_hash_id"].is_unique, "grain broken: expected one row per page"
assert not any(c in FEATURES_W03 + FEATURES_EXTRA for c in ["april_per_day", "april_vs_march_pct", LABEL])
print(f"Frame: {len(frame):,} pages from {frame['client_hash_id'].nunique()} clients | base rate of {LABEL}: {frame[LABEL].mean():.3f}")
print(f"Pages with a full February level: {frame['feb_per_day'].notna().sum():,} ({frame['feb_per_day'].notna().mean():.1%})")

Frame: 94,496 pages from 38 clients | base rate of declined_next_month: 0.516
Pages with a full February level: 69,925 (74.0%)


### The W04 rule and the W04 folds, checked against the W04 receipt

In [5]:
def sliding_with_demand(pages):
    # The W04 baseline rule, copied unchanged. Reads only two March columns
    early, late = pages["early_per_day"], pages["late_per_day"]
    slide_pct = 100 * (1 - late / early.where(early > 0))
    fires = slide_pct.ge(MIN_SLIDE_PCT) & late.ge(MIN_LATE_IMPRESSIONS_PER_DAY)
    return pd.Series(np.where(fires, early - late, 0.0), index=pages.index)


def rank_order(scores, ids):
    # Highest score first. Ties are broken by content_hash_id, the same fixed order as the W04 queue
    return np.lexsort((np.asarray(ids), -np.asarray(scores, dtype=float)))


def queue_metrics(labels, scores, ids):
    y, s = np.asarray(labels), np.asarray(scores, dtype=float)
    top = y[rank_order(s, ids)]
    return {"P@20": top[:20].mean(), "P@50": top[:50].mean(), "P@100": top[:100].mean(),
            "avg precision": average_precision_score(y, s), "NDCG@50": ndcg_score(y[None, :], s[None, :], k=50),
            "ROC AUC": roc_auc_score(y, s)}


frame["rule_score"] = sliding_with_demand(frame[["early_per_day", "late_per_day"]])

# The same GroupKFold as W03 and W04: 5 folds, grouped by client
frame["fold"] = -1
for fold, (_, test_idx) in enumerate(GroupKFold(n_splits=N_FOLDS).split(frame, frame[LABEL], groups=frame["client_hash_id"])):
    frame.loc[test_idx, "fold"] = fold

# The W04 receipt (falls back to the numbers printed in W04 if the file is not there, e.g. in Colab)
receipt_dir = Path("../outputs") if Path.cwd().name == "notebooks" else Path("work/outputs")
w04_path = receipt_dir / "w04_baseline_metrics.json"
w04 = json.loads(w04_path.read_text()) if w04_path.exists() else {
    "data": {"pages": 94496, "clients": 38, "base_rate": 0.5159}, "rule": {"pages_fired": 21855},
    "client_folds": {"rule_p50_per_fold": [0.8, 0.92, 0.88, 0.7, 0.62]}}

rule_p50 = [queue_metrics(g[LABEL], g["rule_score"], g["content_hash_id"])["P@50"] for _, g in frame.groupby("fold")]
same_as_w04 = {
    "pages": len(frame) == w04["data"]["pages"],
    "clients": frame["client_hash_id"].nunique() == w04["data"]["clients"],
    "base rate": round(frame[LABEL].mean(), 4) == w04["data"]["base_rate"],
    "rule picks": int(frame["rule_score"].gt(0).sum()) == w04["rule"]["pages_fired"],
    "rule P@50 on each of the 5 client folds": [round(p, 3) for p in rule_p50] == w04["client_folds"]["rule_p50_per_fold"],
}
for check, ok in same_as_w04.items():
    print(f"{'PASS' if ok else 'FAIL'}  same {check} as W04")
assert all(same_as_w04.values()), "the frame or the folds drifted from W04"
print(f"Read from {w04_path}" if w04_path.exists() else "W04 receipt not found, used the W04 numbers written in the code")

PASS  same pages as W04
PASS  same clients as W04
PASS  same base rate as W04
PASS  same rule picks as W04
PASS  same rule P@50 on each of the 5 client folds as W04
Read from ../outputs/w04_baseline_metrics.json


### The review pool: pages still visible in late March

In [6]:
frame["in_pool"] = frame["late_per_day"] >= MIN_LATE_IMPRESSIONS_PER_DAY      # same 5-a-day bar as the W04 rule
assert frame.loc[frame["rule_score"] > 0, "in_pool"].all(), "every rule pick must be inside the pool"

pool_table = frame.groupby(frame["in_pool"].map({True: "in the pool (5+ a day in late March)", False: "below 5 a day in late March"})).agg(
    pages=(LABEL, "size"), decline_rate=(LABEL, "mean"),
    median_late_march_per_day=("late_per_day", "median"), median_slide_pct=("slide_pct", "median"),
    share_with_0_clicks=("clicks_per_day", lambda x: (x == 0).mean()))
display(pool_table.round(3))

folds = frame.groupby("fold").agg(clients=("client_hash_id", "nunique"), pages=(LABEL, "size"), pages_in_pool=("in_pool", "sum"))
folds["base rate in pool"] = frame[frame["in_pool"]].groupby("fold")[LABEL].mean()
folds["rule picks"] = frame.groupby("fold")["rule_score"].apply(lambda s: int((s > 0).sum()))
folds["rule P@50"] = rule_p50
folds.round(3)

,pages,decline_rate,median_late_march_per_day,median_slide_pct,share_with_0_clicks
in_pool,,,,,
below 5 a day in late March,9904,0.674,3.625,36.141,0.775
in the pool (5+ a day in late March),84592,0.497,33.125,-8.662,0.328


,clients,pages,pages_in_pool,base rate in pool,rule picks,rule P@50
fold,,,,,,
0,1,21238,19352,0.405,5614,0.80
1,7,18314,15659,0.708,5292,0.92
2,9,18314,16853,0.467,5645,0.88
3,10,18317,16858,0.407,2939,0.70
4,11,18313,15870,0.530,2365,0.62


The pool keeps 84,592 of the 94,496 pages (89.5%). The 9,904 pages under the bar declined 67.4% of the time, but their median late-March level is only 3.6 impressions a day and 77.5% of them got no click at all in March. They "decline" mostly because they had already slid away before 31 March. Section 3 shows that a model without the pool goes straight for them.

The pool does not touch the baseline: all 21,855 rule picks are inside it, so the rule's queue and its fold P@50 (0.80, 0.92, 0.88, 0.70, 0.62) are exactly the W04 numbers. The base rate inside the pool is 0.497, from 0.405 in fold 0 (one big client) to 0.708 in fold 1.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

For each fold: train on the pool pages of the other four folds, score the pool pages of the held-out fold, rank them and measure. Same metrics as W04: **P@50** is the main number, with P@20, P@100, average precision, NDCG@50 and ROC AUC as support. Ties are broken by `content_hash_id` like in the W04 queue. The rule has nothing to train, so its fold number is just the rule applied to that fold. The "random order" row is what a random queue gets on average: its P@K equals the fold's base rate.

In [7]:
def add_ctr_gap(train, test):
    # CTR minus the median CTR of the page's position tier. The tier medians come from the training clients only
    tier_train = pd.cut(train["avg_position"], POSITION_TIERS, right=False)
    tier_test = pd.cut(test["avg_position"], POSITION_TIERS, right=False)
    tier_median = train.groupby(tier_train, observed=False)["ctr_pct"].median()
    return (train.assign(ctr_gap=train["ctr_pct"] - tier_train.map(tier_median).astype("float64")),
            test.assign(ctr_gap=test["ctr_pct"] - tier_test.map(tier_median).astype("float64")))


pool = frame[frame["in_pool"]].reset_index(drop=True)
SLIDE_ONLY = "slide only (no model)"
scores = pd.DataFrame({RULE: pool["rule_score"], SLIDE_ONLY: -pool["late_march_share"]})   # out-of-fold scores
fitted, rows = {}, []
for fold in range(N_FOLDS):
    train, test = add_ctr_gap(pool[pool["fold"] != fold], pool[pool["fold"] == fold])
    assert not set(train["client_hash_id"]) & set(test["client_hash_id"]), "a client is on both sides of the split"
    y, ids = test[LABEL].to_numpy(), test["content_hash_id"].to_numpy()
    base = y.mean()
    rows.append({"fold": fold, "method": "random order", "P@20": base, "P@50": base, "P@100": base, "avg precision": base,
                 "NDCG@50": base, "ROC AUC": 0.5})    # expected values for a random queue
    for name in [RULE, SLIDE_ONLY]:
        rows.append({"fold": fold, "method": name, **queue_metrics(y, scores.loc[test.index, name], ids)})
    for name, (cols, make_model) in MODELS.items():
        model = make_model().fit(train[cols], train[LABEL])
        scores.loc[test.index, name] = model.predict_proba(test[cols])[:, 1]
        rows.append({"fold": fold, "method": name, **queue_metrics(y, scores.loc[test.index, name], ids)})
        fitted[(name, fold)] = (model, test)
by_fold = pd.DataFrame(rows)

METHODS = ["random order", RULE, SLIDE_ONLY, *MODELS]
METRICS = ["P@20", "P@50", "P@100", "avg precision", "NDCG@50", "ROC AUC"]
p50 = by_fold.pivot(index="method", columns="fold", values="P@50").loc[METHODS]
p50.columns = [f"fold {c}" for c in p50.columns]
p50["mean"] = p50.mean(axis=1)
p50["folds above the rule"] = [int((p50.loc[m].iloc[:N_FOLDS] > p50.loc[RULE].iloc[:N_FOLDS]).sum()) for m in METHODS]
print(f"Precision@50 on each client fold ({len(pool):,} pages in the pool, models trained on the other 4 folds)")
display(p50.round(3))

table = by_fold.groupby("method")[METRICS].mean().loc[METHODS]
print("Mean over the 5 client folds")
table.round(3)

Precision@50 on each client fold (84,592 pages in the pool, models trained on the other 4 folds)


,fold 0,fold 1,fold 2,fold 3,fold 4,mean,folds above the rule
method,,,,,,,
random order,0.405,0.708,0.467,0.407,0.53,0.503,0
W04 rule (baseline),0.800,0.920,0.880,0.700,0.62,0.784,0
slide only (no model),0.960,0.980,0.780,0.780,0.50,0.800,3
logistic regression,0.980,0.960,0.920,0.820,0.74,0.884,5
gradient boosting,0.880,0.940,0.840,0.920,0.86,0.888,4
gradient boosting + 4 extra,0.980,0.940,0.840,0.840,0.82,0.884,4


Mean over the 5 client folds


,P@20,P@50,P@100,avg precision,NDCG@50,ROC AUC
method,,,,,,
random order,0.503,0.503,0.503,0.503,0.503,0.500
W04 rule (baseline),0.850,0.784,0.810,0.597,0.789,0.616
slide only (no model),0.780,0.800,0.840,0.686,0.787,0.684
logistic regression,0.900,0.884,0.864,0.692,0.888,0.701
gradient boosting,0.890,0.888,0.906,0.699,0.880,0.709
gradient boosting + 4 extra,0.890,0.884,0.894,0.696,0.893,0.705


**What the numbers say.**

- **Logistic regression beats the rule on every fold.** P@50 is 0.884 on average against 0.784 for the rule and it is higher on all 5 folds. Its worst fold (0.74) is still above the rule's worst (0.62). It is also better over the whole queue: average precision 0.692 vs 0.597 and ROC AUC 0.701 vs 0.616.
- **But most of that comes from the slide alone.** Sorting by `late_march_share` with no model already gets average precision 0.686 and P@50 0.800. So the model adds about 8 points of P@50 at the top and very little over the whole ranking. Section 4 shows the extra comes mostly from CTR.
- **Gradient boosting does not earn its place.** P@50 is 0.888 vs 0.884, a difference of one page out of 250. It is a bit better at P@100 (0.906 vs 0.864) and steadier across folds (0.84 to 0.94 vs 0.74 to 0.98), but it beats the rule in 4 folds, not 5.
- **The four extra features add nothing.** Gradient boosting with them gets P@50 0.884 and average precision 0.696, against 0.888 and 0.699 without them. The February level was meant to help the model tell a surge from a real drop, but W04 showed the label rewards both (about 72% "declined" either way), so the model gains nothing from telling them apart.

So **my model is logistic regression on the five W03 features.** It ties with gradient boosting on my main metric, it beats the rule on every fold and I can read every weight. The steadier folds of gradient boosting are worth one more look in W06 under a different split.

### Why the pool matters: the same model without it

In [8]:
def top_k_per_fold(data, score, k=K_MAIN):
    # The first k pages of each fold's queue, stacked (5 folds x k pages)
    parts = []
    for fold in range(N_FOLDS):
        part = data[data["fold"] == fold]
        parts.append(part.iloc[rank_order(np.asarray(score)[part.index], part["content_hash_id"])[:k]])
    return pd.concat(parts)


# Same logistic regression, trained and ranked on ALL 94,496 pages (no pool)
no_pool_score = np.zeros(len(frame))
for fold in range(N_FOLDS):
    train, test = frame[frame["fold"] != fold], frame[frame["fold"] == fold]
    no_pool_score[test.index] = logistic_regression().fit(train[FEATURES_W03], train[LABEL]).predict_proba(test[FEATURES_W03])[:, 1]


def top_shape(top):
    return {"P@50 (mean of folds)": top.groupby("fold")[LABEL].mean().mean(),
            "median late-March impressions per day": top["late_per_day"].median(),
            "share under 5 a day in late March": (top["late_per_day"] < MIN_LATE_IMPRESSIONS_PER_DAY).mean(),
            "share with 0 clicks in March": (top["clicks_per_day"] == 0).mean(),
            "median slide %": top["slide_pct"].median()}


pd.DataFrame({
    "logistic regression, all pages (no pool)": top_shape(top_k_per_fold(frame, no_pool_score)),
    "logistic regression, inside the pool": top_shape(top_k_per_fold(pool, scores[MAIN])),
    f"{RULE}": top_shape(top_k_per_fold(pool, scores[RULE])),
}).T.round(3)

,P@50 (mean of folds),median late-March impressions per day,share under 5 a day in late March,share with 0 clicks in March,median slide %
"logistic regression, all pages (no pool)",0.932,0.156,0.988,0.996,98.652
"logistic regression, inside the pool",0.884,7.531,0.000,0.608,85.317
W04 rule (baseline),0.784,550.625,0.000,0.036,51.799


**Without the pool the model "wins" for the wrong reason.** The same logistic regression trained on all 94,496 pages gets P@50 = 0.932, higher than anything above. But its top 50 is made of pages that are already gone: median late-March level 0.16 impressions a day, 98.8% under the 5-a-day bar, 99.6% with no click in March and a median slide of 98.7%. They count as "declined" because they had already disappeared before the decision moment. A reviewer can't do anything with that list. This is why the pool is part of the split design and why P@50 alone can't pick the winner.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### What the model leans on

In [9]:
AUDIT_VERDICT = {"imp_per_day_log": "OPPOSITE: bigger pages decline less (W04)",
                 "ctr_pct": "CONFIRMED: low CTR for the position (audit)",
                 "avg_position": "MIXED (audit)",
                 "late_march_share": "CONFIRMED: late-month slide (audit)",
                 "gsc_days": "MIXED (audit)"}

weights = pd.DataFrame({fold: fitted[(MAIN, fold)][0][-1].coef_[0] for fold in range(N_FOLDS)}, index=FEATURES_W03)
drops = {}
for name in ["logistic regression", "gradient boosting"]:
    per_fold = []
    for fold in range(N_FOLDS):
        model, test = fitted[(name, fold)]
        result = permutation_importance(model, test[FEATURES_W03], test[LABEL], scoring="average_precision",
                                        n_repeats=5, random_state=SEED)
        per_fold.append(result.importances_mean)
    drops[name] = np.mean(per_fold, axis=0)

lean = pd.DataFrame({
    "LR weight (mean of 5 folds)": weights.mean(axis=1),
    "LR weight, lowest fold": weights.min(axis=1),
    "LR weight, highest fold": weights.max(axis=1),
    "AP drop when shuffled: LR": drops["logistic regression"],
    "AP drop when shuffled: gradient boosting": drops["gradient boosting"],
    "signal verdict from W04": pd.Series(AUDIT_VERDICT),
}).sort_values("AP drop when shuffled: LR", ascending=False)
lean.index.name = "feature"
print("Weights are per 1 standard deviation of the feature. Positive = more likely to decline.")
print("AP drop = how much average precision falls on the held-out clients when that one feature is shuffled (5 repeats, seed 42).")
lean.round(3)

Weights are per 1 standard deviation of the feature. Positive = more likely to decline.
AP drop = how much average precision falls on the held-out clients when that one feature is shuffled (5 repeats, seed 42).


,LR weight (mean of 5 folds),"LR weight, lowest fold","LR weight, highest fold",AP drop when shuffled: LR,AP drop when shuffled: gradient boosting,signal verdict from W04
feature,,,,,,
late_march_share,-0.694,-0.740,-0.666,0.158,0.149,CONFIRMED: late-month slide (audit)
ctr_pct,-0.383,-0.416,-0.342,0.032,0.048,CONFIRMED: low CTR for the position (audit)
avg_position,-0.103,-0.149,-0.032,0.005,0.021,MIXED (audit)
imp_per_day_log,-0.202,-0.253,-0.161,0.002,0.006,OPPOSITE: bigger pages decline less (W04)
gsc_days,-0.003,-0.056,0.066,-0.001,0.010,MIXED (audit)


**The weights agree with the W04 verdicts.** Four of them keep their sign in every fold, only `gsc_days` flips.

1. **`late_march_share` (the slide) carries the model.** Weight −0.69 (−0.74 to −0.67 across folds): the smaller the share of March impressions that came in the second half, the more likely a decline. Shuffling it drops average precision by 0.158. This makes sense and matches the CONFIRMED slide verdict. Part of it is the carry-over W04 measured: a page that just stays at its late-March level is labelled "declined" once its slide is above 34.1%. It is strong but not suspiciously perfect: on its own it gives ROC AUC 0.684, nowhere near the 1.000 of the W03 leak.
2. **`ctr_pct` is the second signal.** Weight −0.38: lower CTR, more decline. Shuffling it costs 0.032 (0.048 for gradient boosting). This matches the CONFIRMED CTR-fix verdict and it is most of what the model adds over the slide alone.
3. **`imp_per_day_log` has a negative weight (−0.20):** bigger pages decline less, the same as the OPPOSITE verdict for volume. It moves the order a little, but shuffling it costs only 0.002.
4. **`avg_position` and `gsc_days` barely matter**, the same as their MIXED verdicts. `gsc_days` even changes sign between folds (−0.06 to +0.07).

Gradient boosting leans on the same two features in the same order. It uses position a bit more (0.021), most likely to read CTR together with position, which is the CTR-fix idea.

### What the top 50 looks like: model vs rule

In [10]:
pool = pool.join(hindsight, on="content_hash_id")       # hindsight joined only now, after every score is fixed
pool["april_lost_per_day"] = pool["imp_per_day"] - pool["april_per_day"]


def top_profile(top):
    with_feb = top[top["feb_per_day"].notna()]
    return {"declined (hindsight)": top[LABEL].mean(),
            "median impressions per day (March)": top["imp_per_day"].median(),
            "median clicks per day (March)": top["clicks_per_day"].median(),
            "share with 0 clicks in March": (top["clicks_per_day"] == 0).mean(),
            "median slide %": top["slide_pct"].median(),
            "surge ended: late March at or above Feb level": (with_feb["late_per_day"] >= with_feb["feb_per_day"]).mean(),
            "pages with a Feb level": len(with_feb),
            "biggest single-client share of a fold's top 50": top.groupby("fold")["client_rank"].agg(lambda s: s.value_counts().iloc[0] / len(s)).mean(),
            "impressions per day lost by April, sum per fold (hindsight)": top.groupby("fold")["april_lost_per_day"].sum().mean()}


# Can the model also find the big losses? Weight its probability by the page's size (no new training)
SIZED = "logistic regression × size"
scores[SIZED] = scores[MAIN] * pool["imp_per_day"]
for fold in range(N_FOLDS):
    test = pool[pool["fold"] == fold]
    by_fold.loc[len(by_fold)] = {"fold": fold, "method": SIZED,
                                 **queue_metrics(test[LABEL], scores.loc[test.index, SIZED], test["content_hash_id"])}

profile = pd.DataFrame({name: top_profile(top_k_per_fold(pool, scores[name])) for name in [RULE, *MODELS, SIZED]})
print(f"Top {K_MAIN} of each of the 5 fold queues (5 x {K_MAIN} = {N_FOLDS * K_MAIN} pages per method)")
profile.round(3)

Top 50 of each of the 5 fold queues (5 x 50 = 250 pages per method)


,W04 rule (baseline),logistic regression,gradient boosting,gradient boosting + 4 extra,logistic regression × size
declined (hindsight),0.784,0.884,0.888,0.884,0.552
median impressions per day (March),872.871,32.145,43.484,49.742,1925.081
median clicks per day (March),0.871,0.000,0.000,0.032,2.548
share with 0 clicks in March,0.036,0.608,0.512,0.464,0.004
median slide %,51.799,85.317,77.697,77.080,8.899
surge ended: late March at or above Feb level,0.492,0.182,0.222,0.199,0.806
pages with a Feb level,189.000,137.000,171.000,161.000,216.000
biggest single-client share of a fold's top 50,0.840,0.572,0.724,0.688,0.876
"impressions per day lost by April, sum per fold (hindsight)",21121.154,1093.680,4082.626,3430.283,17217.317


**The model and the rule fill their top 50 with very different pages.**

| Top 50 of each fold (250 pages) | rule | logistic regression |
|---|---|---|
| declined (hindsight) | 78.4% | 88.4% |
| median impressions per day in March | 873 | 32 |
| share with no click in March | 3.6% | 60.8% |
| surge ended (late March still at or above February) | 49.2% | 18.2% |
| biggest single-client share of a fold's top 50 | 84% | 57% |
| impressions per day lost by April, per fold | 21,121 | 1,094 |

The model is better at "which pages will drop 20%+". Its picks are less often a surge ending and less piled on one client. But they are small pages: half of them under 32 impressions a day and most of them with no clicks. The rule picks big pages, so even with more misses its top 50 lost about 19 times more impressions a day by April. Gradient boosting sits in between (4,083).

**Weighting by size does not fix this.** Multiplying the model's probability by March impressions per day (the "× size" column) brings big pages back, but P@50 falls to 0.552, close to the base rate. 80.6% of its picks with a February level are surges ending and they still lost fewer impressions (17,217) than the rule's picks. For the "where are the big losses" question the W04 rule stays the better tool.

### Where the model is most wrong

In [11]:
# The top 10% of each fold's queue: this is the part a reviewer would actually open
TOP_SHARE = 0.10
in_top = pd.Series(False, index=pool.index)
for fold in range(N_FOLDS):
    part = pool[pool["fold"] == fold]
    order = rank_order(scores.loc[part.index, MAIN], part["content_hash_id"])
    in_top[part.index[order[: int(round(TOP_SHARE * len(part)))]]] = True
top10 = pool[in_top]

volume = pd.cut(top10["imp_per_day"], [0, 10, 30, 100, 300, np.inf], right=False,
                labels=["under 10", "10 to 30", "30 to 100", "100 to 300", "300+"])
by_volume = top10.groupby(volume, observed=True)[LABEL].agg(n="size", precision="mean")
by_volume.index.name = "March impressions per day"

slide = pd.cut(top10["slide_pct"], [-np.inf, 0, 20, 50, 75, 100.001], right=False,
               labels=["rose", "fell 0 to 20%", "fell 20 to 50%", "fell 50 to 75%", "fell 75 to 100%"])
by_slide = top10.groupby(slide, observed=True)[LABEL].agg(n="size", precision="mean")
by_slide.index.name = "change from early to late March"

by_fold_top = top10.groupby("fold").agg(pages=(LABEL, "size"), precision=(LABEL, "mean"))
by_fold_top["P@50"] = p50.loc[MAIN].iloc[:N_FOLDS].to_numpy()
by_fold_top["base rate"] = pool.groupby("fold")[LABEL].mean()

print(f"Logistic regression: top {TOP_SHARE:.0%} of each fold's queue ({len(top10):,} pages), precision = share that declined")
display(by_volume.round(3))
display(by_slide.round(3))
display(by_fold_top.round(3))

# Inside the top 50 of each fold: what separates the false alarms from the hits?
top50 = top_k_per_fold(pool, scores[MAIN])
hits_vs_alarms = top50.groupby(top50[LABEL].map({1: "hit (declined)", 0: "false alarm"})).agg(
    pages=(LABEL, "size"),
    share_missing_part_of_february=("feb_rows", lambda x: (x < FEB_DAYS).mean()),
    median_days_with_search_data=("gsc_days", "median"),
    median_early_march_per_day=("early_per_day", "median"),
    median_late_march_per_day=("late_per_day", "median"),
    median_april_vs_march_pct=("april_vs_march_pct", "median"),
    share_above_march_level_in_april=("april_vs_march_pct", lambda x: (x > 0).mean()))
print(f"Logistic regression, top {K_MAIN} of each fold ({len(top50)} pages). April columns are hindsight")
display(hits_vs_alarms.T.round(3))

# The other side: declined pages that the model ranked in the bottom half of their fold
pct_rank = scores[MAIN].groupby(pool["fold"]).rank(pct=True)
missed = pool[(pool[LABEL] == 1) & (pct_rank < 0.5)]
caught = pool[(pool[LABEL] == 1) & (pct_rank >= 0.5)]
print(f"Declined pages ranked in the bottom half: {len(missed):,} of {int(pool[LABEL].sum()):,} ({len(missed) / pool[LABEL].sum():.1%})")
pd.DataFrame({"declined, bottom half (missed)": [len(missed), (missed["slide_pct"] < 0).mean(), missed["slide_pct"].median(),
                                                  missed["ctr_pct"].median(), missed["april_vs_march_pct"].median()],
              "declined, top half (caught)": [len(caught), (caught["slide_pct"] < 0).mean(), caught["slide_pct"].median(),
                                              caught["ctr_pct"].median(), caught["april_vs_march_pct"].median()]},
             index=["pages", "share that ROSE in late March", "median slide %", "median CTR %", "median April vs March % (hindsight)"]).round(3)

Logistic regression: top 10% of each fold's queue (8,459 pages), precision = share that declined


,n,precision
March impressions per day,,
under 10,1555,0.709
10 to 30,4046,0.792
30 to 100,2241,0.845
100 to 300,512,0.867
300+,105,0.876


,n,precision
change from early to late March,,
rose,15,0.733
fell 0 to 20%,505,0.735
fell 20 to 50%,3366,0.737
fell 50 to 75%,3661,0.835
fell 75 to 100%,912,0.898


,pages,precision,P@50,base rate
fold,,,,
0,1935,0.722,0.98,0.405
1,1566,0.925,0.96,0.708
2,1685,0.784,0.92,0.467
3,1686,0.772,0.82,0.407
4,1587,0.801,0.74,0.530


Logistic regression, top 50 of each fold (250 pages). April columns are hindsight


declined_next_month,false alarm,hit (declined)
pages,29.000,221.000
share_missing_part_of_february,0.690,0.421
median_days_with_search_data,29.000,31.000
median_early_march_per_day,64.333,56.200
median_late_march_per_day,7.938,7.438
median_april_vs_march_pct,55.394,-82.526
share_above_march_level_in_april,0.759,0.000


Declined pages ranked in the bottom half: 15,219 of 42,071 (36.2%)


,"declined, bottom half (missed)","declined, top half (caught)"
pages,15219.000,26852.000
share that ROSE in late March,0.826,0.227
median slide %,-43.257,24.290
median CTR %,0.187,0.061
median April vs March % (hindsight),-43.107,-49.808


**Where the model is most wrong:**

- **Small pages.** In the top 10% of each fold's queue, precision climbs with size: 70.9% for pages under 10 impressions a day up to 87.6% for 300+. Most of the top 10% is small pages (5,601 of 8,459 are under 30 a day). At 10 impressions a day a 20% move is two impressions, so part of what the model ranks there is noise.
- **Moderate slides.** Pages that fell 20 to 50% inside March are right 73.7% of the time, steep slides of 75%+ are right 89.8% of the time. The model is surest where the carry-over does most of the work.
- **Young pages with a short dip (the false alarms).** The top 50 of each fold hold 29 false alarms out of 250 pages. 69.0% of them are missing part of February (new pages, or pages that were quiet), against 42.1% of the hits. 75.9% of them ended April above their March level (median +55.4%). So the typical false alarm is a newer page that dropped out for a while in late March and came back in April.
- **Pages that were growing (the misses).** 15,219 of the 42,071 pool pages that declined (36.2%) sit in the bottom half of their fold's queue. 82.6% of these rose in late March (median slide −43%). They fell in April after growing in March and nothing in March shows that coming. Some of them are a late-March surge that ended in April (case 3 below).
- **By fold**, the weakest P@50 is fold 4 (0.74, 11 clients). Over the full top 10% the lowest precision is fold 0 (72.2%), the single biggest client, but that fold also has the lowest base rate (0.405).

### Three concrete wrong cases

In [12]:
pool["fold_rank"] = scores[MAIN].groupby(pool["fold"]).rank(ascending=False, method="first").astype(int)
top50 = top50.join(pool["fold_rank"])
false_pos = top50[top50[LABEL] == 0]
case_1 = false_pos.sort_values(["fold_rank", "content_hash_id"]).index[0]                          # highest-ranked miss
case_2 = false_pos.drop(index=case_1).sort_values("imp_per_day", ascending=False).index[0]          # biggest page among the misses
case_3 = missed.sort_values("april_lost_per_day", ascending=False).index[0]                         # biggest loss the model ranked low
cases = pool.loc[[case_1, case_2, case_3]].copy()
cases["model score"] = scores.loc[cases.index, MAIN]
cases.index = ["case 1: top-50 false alarm", "case 2: biggest top-50 false alarm", "case 3: biggest missed decline"]
case_cols = ["fold", "client_rank", "fold_rank", "model score", "feb_per_day", "early_per_day", "late_per_day", "slide_pct",
             "early_peak_share", "clicks_per_day", "ctr_pct", "avg_position", "gsc_days", "april_per_day", "april_vs_march_pct", LABEL]
display(cases[case_cols].rename(columns={"april_per_day": "april_per_day (hindsight)",
                                         "april_vs_march_pct": "april_vs_march_% (hindsight)",
                                         LABEL: f"{LABEL} (hindsight)"}).T.round(3))

# Daily impressions of the three pages: February to April (April is hindsight, read only now)
con.register("case_pages", pd.DataFrame({"content_hash_id": pool.loc[[case_1, case_2, case_3], "content_hash_id"].to_numpy(),
                                         "case": [1, 2, 3]}))
daily = con.sql(f'''
    SELECT c.case, f.report_date, f.gsc_impressions
    FROM read_parquet(['{REL}/fact_content_daily_performance/month={PRE_MONTH}/*.parquet',
                       '{REL}/fact_content_daily_performance/month={FEATURE_MONTH}/*.parquet',
                       '{REL}/fact_content_daily_performance/month={LABEL_MONTH}/*.parquet'])  f
    JOIN case_pages c USING (content_hash_id)
''').df()
daily["report_date"] = pd.to_datetime(daily["report_date"])
daily["week"] = daily["report_date"].dt.to_period("W-SUN").dt.start_time.dt.strftime("%d %b")
weekly = daily.groupby(["case", "week"])["gsc_impressions"].mean().unstack("week").sort_index()   # fixed row order
weekly = weekly[sorted(weekly.columns, key=lambda w: pd.Timestamp(f"{w} 2026"))]
print("Average impressions per day, by week starting (February to April; April is hindsight)")
with pd.option_context("display.max_columns", 20):
    display(weekly.round(0))

,case 1: top-50 false alarm,case 2: biggest top-50 false alarm,case 3: biggest missed decline
fold,4.000,2.000,2.000
client_rank,20.000,8.000,3.000
fold_rank,2.000,30.000,16697.000
model score,0.900,0.871,0.089
feb_per_day,NaN,NaN,3222.250
early_per_day,134.333,517.667,481.533
late_per_day,8.812,47.188,12823.812
slide_pct,93.440,90.885,-2563.120
early_peak_share,0.346,0.088,0.193
clicks_per_day,0.032,0.355,0.774


Average impressions per day, by week starting (February to April; April is hindsight)


week,26 Jan,02 Feb,09 Feb,16 Feb,23 Feb,02 Mar,09 Mar,16 Mar,23 Mar,30 Mar,06 Apr,13 Apr,20 Apr,27 Apr
case,,,,,,,,,,,,,,
1,NaN,NaN,NaN,0.0,219.0,216.0,18.0,16.0,4.0,73.0,162.0,145.0,97.0,116.0
2,NaN,NaN,NaN,0.0,85.0,575.0,492.0,87.0,19.0,701.0,1748.0,1857.0,1010.0,546.0
3,50.0,44.0,621.0,11669.0,589.0,307.0,683.0,300.0,28352.0,4017.0,2701.0,616.0,359.0,278.0


**Three wrong cases and why they are hard:**

| Case | What the model saw (up to 31 March) | What happened (April, hindsight) | Why it is hard |
|---|---|---|---|
| **1. False alarm, #2 in fold 4** (score 0.90) | A young page with rows only from mid February. 134 impressions a day in early March, then 8.8 a day in late March (−93%) and almost no clicks | It came back in the week of 30 March: 127 a day in April, +82% vs March | Inside March, a page that switched off and a page that is about to switch back on look the same. The return started around the decision moment |
| **2. Biggest false alarm in the top 50, #30 in fold 2** (score 0.87) | Also young (rows from mid February). 518 a day in early March at position 2.2, then 47 a day in late March (−91%) | Also back in the week of 30 March, then 1,313 a day in April, +378% vs March | The same pattern at a bigger size. A two week dip on a new page is common and the model has no way to see that it is temporary. This is the "young page with a short dip" error from the table above |
| **3. Biggest missed decline, #16,697 of 16,853 in fold 2** (score 0.09) | Rose from 482 to 12,824 a day (the week of 23 March averaged about 28,000 a day). CTR 0.011% and an odd position of 0.67 | Fell back to 1,678 a day in April (−75.5% vs the March average) | The model reads "rising fast" as safe. But this page lives on spikes: February had one too (the week of 16 February averaged about 11,700 a day, the other weeks under 700). The "decline" is the spike ending, which the label counts and a refresh could not prevent. This is the W04 surge problem showing up as a miss instead of a hit |

Cases 1 and 2 are the main way the model goes wrong at the top. Case 3 is as much a label problem as a model problem.

### Final table: model vs baseline

In [13]:
final_methods = ["random order", RULE, SLIDE_ONLY, *MODELS, SIZED]
table = by_fold.groupby("method")[METRICS].mean()                      # now includes the size-weighted row
p50 = by_fold.pivot(index="method", columns="fold", values="P@50")
p50["folds above the rule"] = [int((p50.loc[m, range(N_FOLDS)] > p50.loc[RULE, range(N_FOLDS)]).sum()) for m in p50.index]
final = table.loc[final_methods, ["P@20", "P@50", "P@100", "avg precision", "ROC AUC"]].copy()
final["P@50 lowest fold"] = p50.loc[final_methods].iloc[:, :N_FOLDS].min(axis=1)
final["P@50 highest fold"] = p50.loc[final_methods].iloc[:, :N_FOLDS].max(axis=1)
final["folds where P@50 beats the rule"] = p50.loc[final_methods, "folds above the rule"].astype("float64")
final.loc[RULE, "folds where P@50 beats the rule"] = np.nan
for name in [RULE, SLIDE_ONLY, *MODELS, SIZED]:
    top = top_k_per_fold(pool, scores[name])
    final.loc[name, "top 50: median impressions per day"] = top["imp_per_day"].median()
    final.loc[name, "top 50: impressions per day lost by April (hindsight)"] = top.groupby("fold")["april_lost_per_day"].sum().mean()
print(f"Same {len(pool):,} pages (W03 frame, still visible in late March), same 5 client folds as W04, mean over folds")
print(f"Base rate in the pool: {pool[LABEL].mean():.3f}. My model: {MAIN}.")
final.round(3)

Same 84,592 pages (W03 frame, still visible in late March), same 5 client folds as W04, mean over folds
Base rate in the pool: 0.497. My model: logistic regression.


,P@20,P@50,P@100,avg precision,ROC AUC,P@50 lowest fold,P@50 highest fold,folds where P@50 beats the rule,top 50: median impressions per day,top 50: impressions per day lost by April (hindsight)
method,,,,,,,,,,
random order,0.503,0.503,0.503,0.503,0.500,0.405,0.708,0.0,NaN,NaN
W04 rule (baseline),0.850,0.784,0.810,0.597,0.616,0.620,0.920,NaN,872.871,21121.154
slide only (no model),0.780,0.800,0.840,0.686,0.684,0.500,0.980,3.0,48.984,270.143
logistic regression,0.900,0.884,0.864,0.692,0.701,0.740,0.980,5.0,32.145,1093.680
gradient boosting,0.890,0.888,0.906,0.699,0.709,0.840,0.940,4.0,43.484,4082.626
gradient boosting + 4 extra,0.890,0.884,0.894,0.696,0.705,0.820,0.980,4.0,49.742,3430.283
logistic regression × size,0.520,0.552,0.548,0.517,0.527,0.320,0.740,0.0,1925.081,17217.317


**Model vs baseline in one line:** on the same 84,592 pages, the same five client folds and the same metric, my logistic regression finds more pages that go on to lose 20%+ of their impressions (P@50 0.884 vs 0.784, higher on all 5 folds, average precision 0.692 vs 0.597). This is a measured result on held-out clients. It does not mean the model is the better queue on its own.

**What the errors look like.** The model's top 50 is mostly small pages that have already slid hard, which is where the label is easiest to hit, partly by carry-over. Its false alarms are young pages with a short dip that came back in April. Its misses are pages that grew in March and fell in April, including spikes that ended. The rule's errors are the opposite: big pages and often a surge ending.

**What I take into the next weeks:**
- The two tools answer different questions. The rule finds where the big losses are (about 19 times more impressions lost in its top 50). The model is better at telling which pages will drop. For the W07 playbook I will look at using both instead of replacing one with the other.
- The label rewards carry-over and surges ending. In W06 I will re-check the model under a time-aware split and with a label that is not fooled by carry-over, for example April compared with late March, so a page only counts if it kept falling after the decision moment.
- Gradient boosting was steadier across folds. If that holds in W06, it is the one to switch to.

In [14]:
# The run's receipt: small, no page ids, safe to commit (CSV files stay out of git by design)
receipt = {
    "notebook": "work/notebooks/w05_model.ipynb",
    "run_date": str(date.today()),
    "versions": VERSIONS,
    "seed": SEED,
    "data": {"frame": "W03 / W04 page frame", "pages": int(len(frame)), "clients": int(frame["client_hash_id"].nunique()),
             "review_pool": f"late-March impressions per day >= {MIN_LATE_IMPRESSIONS_PER_DAY} (same bar as the W04 rule)",
             "pool_pages": int(len(pool)), "pool_base_rate": round(float(pool[LABEL].mean()), 4)},
    "split": f"GroupKFold(n_splits={N_FOLDS}) on client_hash_id, the same folds as W04 (checked)",
    "my_model": {"name": MAIN, "features": MODELS[MAIN][0]},
    "models": {name: {"features": cols} for name, (cols, _) in MODELS.items()},
    "mean_over_folds": {m: {k: round(float(v), 4) for k, v in row.items()} for m, row in table.iterrows()},
    "p50_per_fold": {m: [round(float(v), 3) for v in row.iloc[:N_FOLDS]] for m, row in p50.iterrows()},
    "top50_profile": {m: {k: round(float(v), 4) for k, v in col.items()} for m, col in profile.items()},
}
OUT_DIR = Path("../outputs") if Path.cwd().name == "notebooks" else Path("work/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
receipt_path = OUT_DIR / "w05_model_metrics.json"
receipt_path.write_text(json.dumps(receipt, indent=2))
print(f"Wrote {receipt_path}")

Wrote ../outputs/w05_model_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**Checked against "what done looks like":**

- [x] Compares against the baseline on the same split: the same five client folds as W04, checked against the W04 receipt (same pages, base rate and rule P@50 per fold)
- [x] Valid split: grouped by client, so every number is measured on clients the model never saw. Scaling and CTR tier medians are learned on the training folds only
- [x] Method choice explained: classifier probability → ranking → P@50, logistic regression first, gradient boosting as the check and a simplicity rule to pick between them
- [x] Useful metrics: P@20, P@50, P@100, average precision, NDCG@50 and ROC AUC, per fold and as a mean, next to a random floor, the rule and a slide-only sort
- [x] Features and errors interpreted: weights checked against the W04 verdicts, permutation importance on held-out clients, the top 50 profile, where precision drops and three concrete wrong cases
- [x] Does not reward complexity alone: gradient boosting and the extra features did not earn their place, the no-pool "win" is shown and rejected and the size trade-off is reported
- [x] Reproducible: seed 42 everywhere, library versions printed at the top, receipt written to `work/outputs/w05_model_metrics.json`